<a href="https://colab.research.google.com/github/fadzankerim/Network-Intrusion-Detection-System/blob/cleaning-transformation/Cleaning_Transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import json
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_class_weight
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(' Biblioteke učitane!')

 Biblioteke učitane!


In [2]:
# ID podijeljenog foldera na Google Drive-u
GDRIVE_FOLDER_ID = '1wqUUXHiIzMHGnOgqezpQDQyDzJ8fzXOm'
GDRIVE_FOLDER_URL = f'https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}'

# Pokušaj montiranja Google Drive-a
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print(f'ℹ️ Google Drive mount preskočen ili nije na Colab-u: {e}')

def resolve_dataset_directory(folder_id=GDRIVE_FOLDER_ID):
    candidate_paths = [
        Path('/content/Dataset'),
        Path('/content/drive/MyDrive/MPVI/Dataset'),
        Path('/content/drive/MyDrive/Dataset'),
        Path('./Dataset'),
        Path('../Dataset')
    ]
    for p in candidate_paths:
        if p.exists() and len(list(p.glob('**/UNSW_NB15_training-set.csv'))) > 0:
            found_file = list(p.glob('**/UNSW_NB15_training-set.csv'))[0]
            ds_root = found_file.parent.parent if found_file.parent.name == 'Training and Testing Sets' else found_file.parent
            print(f'✅ Dataset pronađen na: {ds_root}')
            return str(ds_root)

    drive_root = Path('/content/drive/MyDrive')
    if drive_root.exists():
        found = list(drive_root.glob('**/UNSW_NB15_training-set.csv'))
        if found:
            ds_root = found[0].parent.parent if found[0].parent.name == 'Training and Testing Sets' else found[0].parent
            print(f'✅ Dataset pronađen na Drive-u: {ds_root}')
            return str(ds_root)

    print(' Preuzimam dataset sa Google Drive linka...')
    try:
        import gdown
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
    download_dir = Path('/content/Dataset')
    download_dir.mkdir(parents=True, exist_ok=True)
    try:
        gdown.download_folder(id=folder_id, output=str(download_dir), quiet=False, use_cookies=False)
    except Exception:
        gdown.download_folder(url=GDRIVE_FOLDER_URL, output=str(download_dir), quiet=False)
    found = list(download_dir.glob('**/UNSW_NB15_training-set.csv'))
    if found:
        ds_root = found[0].parent.parent if found[0].parent.name == 'Training and Testing Sets' else found[0].parent
        return str(ds_root)
    return str(download_dir)

DATASET_PATH = resolve_dataset_directory()
if not DATASET_PATH.endswith('/'):
    DATASET_PATH += '/'

train_matches = list(Path(DATASET_PATH).glob('**/UNSW_NB15_training-set.csv'))
test_matches = list(Path(DATASET_PATH).glob('**/UNSW_NB15_testing-set.csv'))
TRAIN_PATH = str(train_matches[0]) if train_matches else DATASET_PATH + 'Training and Testing Sets/UNSW_NB15_training-set.csv'
TEST_PATH = str(test_matches[0]) if test_matches else DATASET_PATH + 'Training and Testing Sets/UNSW_NB15_testing-set.csv'

print(f'\n📂 DATASET_PATH: {DATASET_PATH}')
print(f'📄 TRAIN_PATH:   {TRAIN_PATH}')
print(f'📄 TEST_PATH:    {TEST_PATH}')

Mounted at /content/drive
✅ Dataset pronađen na: /content/drive/MyDrive/MPVI/Dataset

📂 DATASET_PATH: /content/drive/MyDrive/MPVI/Dataset/
📄 TRAIN_PATH:   /content/drive/MyDrive/MPVI/Dataset/Training and Testing Sets/UNSW_NB15_training-set.csv
📄 TEST_PATH:    /content/drive/MyDrive/MPVI/Dataset/Training and Testing Sets/UNSW_NB15_testing-set.csv


In [4]:
# Učitavanje training i testing seta
df_train_raw = pd.read_csv(TRAIN_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

df_train = df_train_raw.copy()
df_test  = df_test_raw.copy()

# Označavanje izvora
df_train['source'] = 'train'
df_test['source']  = 'test'

print(f'Training: {df_train.shape[0]:,} zapisa, {df_train.shape[1]} kolona')
print(f'Testing:  {df_test.shape[0]:,} zapisa, {df_test.shape[1]} kolona')
print(f'Ukupno:   {df_train.shape[0] + df_test.shape[0]:,} zapisa')
display(df_train.head())

Training: 175,341 zapisa, 46 kolona
Testing:  82,332 zapisa, 46 kolona
Ukupno:   257,673 zapisa


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,sloss,dloss,sinpkt,dinpkt,sjit,djit,swin,stcpb,dtcpb,dwin,tcprtt,synack,ackdat,smean,dmean,trans_depth,response_body_len,ct_srv_src,ct_state_ttl,ct_dst_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label,source
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,252,254,14158.942380,8495.365234,0,0,24.295600,8.375000,30.177547,11.830604,255,621772692,2202533631,255,0.000000,0.000000,0.000000,43,43,0,0,1,0,1,1,1,1,0,0,0,1,1,0,Normal,0,train
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,62,252,8395.112305,503571.312500,2,17,49.915000,15.432865,61.426934,1387.778330,255,1417884146,3077387971,255,0.000000,0.000000,0.000000,52,1106,0,0,43,1,1,1,1,2,0,0,0,1,6,0,Normal,0,train
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,62,252,1572.271851,60929.230470,1,6,231.875571,102.737203,17179.586860,11420.926230,255,2116150707,2963114973,255,0.111897,0.061458,0.050439,46,824,0,0,7,1,2,1,1,3,0,0,0,2,6,0,Normal,0,train
3,4,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,62,252,2740.178955,3358.622070,1,3,152.876547,90.235726,259.080172,4991.784669,255,1107119177,1047442890,255,0.000000,0.000000,0.000000,52,64,0,0,1,1,2,1,1,3,1,1,0,2,1,0,Normal,0,train
4,5,0.449454,tcp,-,FIN,10,6,534,268,33.373826,254,252,8561.499023,3987.059814,2,1,47.750333,75.659602,2415.837634,115.807000,255,2436137549,1977154190,255,0.128381,0.071147,0.057234,53,45,0,0,43,1,2,2,1,40,0,0,0,2,39,0,Normal,0,train


In [5]:
# Sačuvaj originalne oblike za poređenje u rezimeu
original_train_shape = df_train.shape
original_test_shape  = df_test.shape

TARGET = 'label'
TARGET_MULTI = 'attack_cat'
CAT_FEATURES = ['proto', 'service', 'state']
BINARY_FEATURES = ['is_ftp_login', 'is_sm_ips_ports']

print(f'Originalni train oblik: {original_train_shape}')
print(f'Originalni test oblik:  {original_test_shape}')

Originalni train oblik: (175341, 46)
Originalni test oblik:  (82332, 46)


In [6]:
# 'id' je samo redni broj zapisa i ne nosi informaciju o saobraćaju
DROP_COLUMNS = ['id']

df_train = df_train.drop(columns=DROP_COLUMNS, errors='ignore')
df_test  = df_test.drop(columns=DROP_COLUMNS, errors='ignore')

print(f'✅ Uklonjene kolone: {DROP_COLUMNS}')
print(f'Train: {df_train.shape} | Test: {df_test.shape}')

✅ Uklonjene kolone: ['id']
Train: (175341, 45) | Test: (82332, 45)


In [7]:
def report_missing(df, name):
    num = df.select_dtypes(include=[np.number])
    n_nan = df.isnull().sum().sum()
    n_inf = np.isinf(num).sum().sum()
    neg_cols = [c for c in num.columns if (num[c] < 0).any()]
    print(f'{name}: NaN = {n_nan}, inf = {n_inf}, kolone sa negativnim vrijednostima = {neg_cols or "nema"}')

report_missing(df_train, 'Train')
report_missing(df_test, 'Test')

Train: NaN = 0, inf = 0, kolone sa negativnim vrijednostima = nema
Test: NaN = 0, inf = 0, kolone sa negativnim vrijednostima = nema


In [8]:
# EDA je potvrdila da nema NaN vrijednosti, ali zaštita ostaje za slučaj novih podataka.
# inf -> NaN, a NaN se popunjava medijanom/modom izračunatim SAMO na train skupu.
numeric_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c != TARGET]

for d in (df_train, df_test):
    d[numeric_cols] = d[numeric_cols].replace([np.inf, -np.inf], np.nan)

NUMERIC_FILL_VALUES = df_train[numeric_cols].median().to_dict()
CATEGORICAL_FILL_VALUES = {c: df_train[c].mode()[0] for c in CAT_FEATURES}

for d in (df_train, df_test):
    d[numeric_cols] = d[numeric_cols].fillna(NUMERIC_FILL_VALUES)
    d[CAT_FEATURES] = d[CAT_FEATURES].fillna(CATEGORICAL_FILL_VALUES)

print(f'✅ Preostale NaN vrijednosti – train: {df_train.isnull().sum().sum()}, test: {df_test.isnull().sum().sum()}')

✅ Preostale NaN vrijednosti – train: 0, test: 0


In [9]:
def clean_categoricals(df):
    df = df.copy()
    for col in CAT_FEATURES:
        df[col] = df[col].astype(str).str.strip().str.lower()
    df['service'] = df['service'].replace({'-': 'none', '': 'none'})

    if TARGET_MULTI in df.columns:
        df[TARGET_MULTI] = df[TARGET_MULTI].fillna('Normal').astype(str).str.strip()
        df[TARGET_MULTI] = df[TARGET_MULTI].replace({'': 'Normal', 'Backdoors': 'Backdoor'})
    return df

df_train = clean_categoricals(df_train)
df_test  = clean_categoricals(df_test)

for col in CAT_FEATURES:
    print(f'{col}: train {df_train[col].nunique()} kategorija, test {df_test[col].nunique()} kategorija')
print(f'\nKategorije napada: {sorted(df_train[TARGET_MULTI].unique())}')

proto: train 133 kategorija, test 131 kategorija
service: train 13 kategorija, test 13 kategorija
state: train 9 kategorija, test 7 kategorija

Kategorije napada: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


In [10]:
print('Prije ispravke:')
for col in BINARY_FEATURES:
    print(f'  {col}: train {sorted(df_train[col].unique())}, test {sorted(df_test[col].unique())}')

for d in (df_train, df_test):
    for col in BINARY_FEATURES:
        d[col] = (pd.to_numeric(d[col], errors='coerce').fillna(0) > 0).astype(int)
    d['ct_ftp_cmd'] = pd.to_numeric(d['ct_ftp_cmd'], errors='coerce').fillna(0).astype(int)
    d[TARGET] = d[TARGET].astype(int)

print('\nPoslije ispravke:')
for col in BINARY_FEATURES + [TARGET]:
    print(f'  {col}: train {sorted(df_train[col].unique())}, test {sorted(df_test[col].unique())}')

Prije ispravke:
  is_ftp_login: train [np.int64(0), np.int64(1), np.int64(2), np.int64(4)], test [np.int64(0), np.int64(1), np.int64(2)]
  is_sm_ips_ports: train [np.int64(0), np.int64(1)], test [np.int64(0), np.int64(1)]

Poslije ispravke:
  is_ftp_login: train [np.int64(0), np.int64(1)], test [np.int64(0), np.int64(1)]
  is_sm_ips_ports: train [np.int64(0), np.int64(1)], test [np.int64(0), np.int64(1)]
  label: train [np.int64(0), np.int64(1)], test [np.int64(0), np.int64(1)]


In [11]:
for name, d in (('Train', df_train), ('Test', df_test)):
    bad_normal = ((d[TARGET] == 0) & (d[TARGET_MULTI] != 'Normal')).sum()
    bad_attack = ((d[TARGET] == 1) & (d[TARGET_MULTI] == 'Normal')).sum()
    print(f'{name}: label=0 ali napad: {bad_normal} | label=1 ali Normal: {bad_attack}')

# Binarni target je izvor istine: ako postoje nekonzistentnosti, attack_cat se usklađuje sa label
for d in (df_train, df_test):
    d.loc[(d[TARGET] == 0), TARGET_MULTI] = 'Normal'
    d.loc[(d[TARGET] == 1) & (d[TARGET_MULTI] == 'Normal'), TARGET_MULTI] = 'Unknown'

Train: label=0 ali napad: 0 | label=1 ali Normal: 0
Test: label=0 ali napad: 0 | label=1 ali Normal: 0


In [12]:
FEATURE_COLS_RAW = [c for c in df_train.columns if c not in (TARGET, TARGET_MULTI, 'source')]

dup_full = df_train.duplicated(subset=FEATURE_COLS_RAW + [TARGET], keep='first').sum()
label_per_group = df_train.groupby(FEATURE_COLS_RAW, dropna=False)[TARGET].transform('nunique')
conflicting_mask = label_per_group > 1

print(f'Train zapisa:                          {len(df_train):,}')
print(f'Duplikati (feature-i + label):         {dup_full:,} ({dup_full/len(df_train)*100:.2f}%)')
print(f'Zapisi u kontradiktornim grupama:      {conflicting_mask.sum():,}')
print(f'Duplikati unutar test seta (zadržani): {df_test.duplicated(subset=FEATURE_COLS_RAW + [TARGET]).sum():,}')

Train zapisa:                          175,341
Duplikati (feature-i + label):         74,072 (42.24%)
Zapisi u kontradiktornim grupama:      940
Duplikati unutar test seta (zadržani): 28,380


In [14]:
# Prekidač za eksperimente: False = zadržati originalni train skup (radi poređenja rezultata modela)
DROP_DUPLICATES = True

train_before = len(df_train)
label_dist_before = df_train[TARGET].value_counts(normalize=True)

if DROP_DUPLICATES:
    # 1) Uklanjanje kontradiktornih zapisa (isti feature-i, različit label)
    df_train = df_train[~conflicting_mask].copy()
    # 2) Uklanjanje duplikata
    df_train = df_train.drop_duplicates(subset=FEATURE_COLS_RAW + [TARGET], keep='first').reset_index(drop=True)

label_dist_after = df_train[TARGET].value_counts(normalize=True)

print(f'✅ Uklonjeno {train_before - len(df_train):,} zapisa iz train skupa')
print(f'Train: {train_before:,} -> {len(df_train):,}')
print(f'\nUdio napada u train skupu: {label_dist_before[1]*100:.2f}% -> {label_dist_after[1]*100:.2f}%')
print(f'Udio napada u test skupu:  {df_test[TARGET].mean()*100:.2f}%')

print('\nDistribucija attack_cat u train skupu nakon čišćenja:')
display(df_train[TARGET_MULTI].value_counts().to_frame('broj zapisa'))

✅ Uklonjeno 715 zapisa iz train skupa
Train: 100,811 -> 100,096

Udio napada u train skupu: 48.75% -> 48.57%
Udio napada u test skupu:  55.06%

Distribucija attack_cat u train skupu nakon čišćenja:


,broj zapisa
attack_cat,
Normal,51483
Exploits,18532
Fuzzers,14643
Reconnaissance,6344
Generic,3894
DoS,2967
Shellcode,1074
Analysis,534
Backdoor,506
